
# P2：终端惩罚系数 $\lambda$ 敏感性分析与论文图总集

> **目标**：基于题目原始附件、预处理后的输入数据以及当前 `q2.py / q2_with_penalty.py`，**完全重新计算**第二问中软终端惩罚模型的参数敏感性分析，并自动生成适合论文挑选的图池。  
> **原则**：不读取任何旧的敏感性结果、旧图、旧中间 CSV；所有结果都由本 Notebook 当次重跑生成。

---

## 一、固定输入文件（必须与本 Notebook 位于同一目录）

```text
附件1.xlsx
附件2.xlsx
q2_actual_load_pv.csv
q2_fixed_prices.csv
result2_template.xlsx
q2.py
q2_with_penalty.py
```

---

## 二、本 Notebook 的模型口径

### 1. 预测层（冻结）
使用当前 `q2.py` 中已经确认的滚动 PCA + Ridge 预测模型：

\[
W=60,\quad K_L=3,\quad K_P=2,\quad \alpha_L=0.3,\quad \alpha_P=10.
\]

### 2. 调度层（本 Notebook 的研究对象）
本 Notebook 研究**软终端惩罚参数**

\[
\lambda
\]

对最终运行效果的影响。每日计划优化目标写为：

\[
\min\; C_{\text{plan}} + \lambda\,|E_{24}-E^\star|,
\qquad E^\star = 6000\ \text{kWh}.
\]

其中：
- \(C_{\text{plan}}\) 为当日计划购电费用；
- \(E_{24}\) 为当日 24 点时的储能量；
- 软惩罚项用于缓解逐日优化的**终端短视效应**；
- **题目定义下的实际总费用**仍然按

\[
C_{\text{actual}} = C_{\text{planned purchase}} + C_{\text{emergency}}
\]

统计，**惩罚项不记入最终实际电费**。

---

## 三、优秀论文常见图型及本 Notebook 的采用方式

国赛优秀论文中，参数敏感性分析常见并且有效的图型包括：

1. **单参数折线图**：用于展示“参数变化 $\to$ 核心指标变化”；
2. **费用分解多折线图**：用于解释总费用变化来源；
3. **柱线组合图**：在一张图中同时展示“成本 + 机制量”；
4. **箱线图**：展示日度分布、波动与稳定性；
5. **热力图**：用于二维参数联合扫描的全局展示；
6. **机制对比曲线图**：展示代表性日期上不同参数下的运行轨迹差异。

因此，本 Notebook 会为每一种图型都至少生成一类候选图，并分别输出：

- `figures_lambda_A_standard/`：规范论文版（稳重、适合正文）；
- `figures_lambda_B_enhanced/`：增强展示版（强调重点、方便择优）。


In [14]:

from __future__ import annotations

from pathlib import Path
import importlib.util
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import MaxNLocator

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 240)

# ============================================================
# 固定路径：只认与 Notebook 同目录的 7 个输入文件
# ============================================================
ROOT = Path.cwd()
ACTUAL_XLSX = ROOT / '附件2.xlsx'
PRICE_XLSX = ROOT / '附件1.xlsx'
ACTUAL_CSV = ROOT / 'q2_actual_load_pv.csv'
PRICE_CSV = ROOT / 'q2_fixed_prices.csv'
TEMPLATE_XLSX = ROOT / 'result2_template.xlsx'
Q2_PY = ROOT / 'q2.py'
Q2_PENALTY_PY = ROOT / 'q2_with_penalty.py'

REQUIRED = [ACTUAL_XLSX, PRICE_XLSX, ACTUAL_CSV, PRICE_CSV, TEMPLATE_XLSX, Q2_PY, Q2_PENALTY_PY]
for p in REQUIRED:
    if not p.exists():
        raise FileNotFoundError(f'缺少必要文件：{p.name}，请确认其与 Notebook 位于同一目录。')

OUT = ROOT / 'fresh_outputs_lambda'
FIG_A = ROOT / 'figures_lambda_A_standard'
FIG_B = ROOT / 'figures_lambda_B_enhanced'
OUT.mkdir(exist_ok=True)
FIG_A.mkdir(exist_ok=True)
FIG_B.mkdir(exist_ok=True)

# 软终端惩罚的参考终端储能
E_TARGET = 6000.0

# 推荐扫描集合：轻量但足够说明问题
LAMBDA_GRID = [0.00, 0.10, 0.20, 0.30, 0.45, 0.60, 0.80, 1.00]
COMPARE_LAMBDAS = [0.00, 0.45, 1.00]  # 代表性机制对比

print('当前目录：', ROOT)
print('输出目录：', OUT)
print('A类图目录：', FIG_A)
print('B类图目录：', FIG_B)
print('lambda 扫描集合：', LAMBDA_GRID)

当前目录： e:\桌面\新建文件夹\P2_SOFT_PENALTY_PACKAGE
输出目录： e:\桌面\新建文件夹\P2_SOFT_PENALTY_PACKAGE\fresh_outputs_lambda
A类图目录： e:\桌面\新建文件夹\P2_SOFT_PENALTY_PACKAGE\figures_lambda_A_standard
B类图目录： e:\桌面\新建文件夹\P2_SOFT_PENALTY_PACKAGE\figures_lambda_B_enhanced
lambda 扫描集合： [0.0, 0.1, 0.2, 0.3, 0.45, 0.6, 0.8, 1.0]


In [15]:

# ---------- 中文字体与统一作图风格 ----------
def setup_chinese_font():
    candidates = ['Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'Source Han Sans SC', 'WenQuanYi Zen Hei', 'Arial Unicode MS']
    all_names = {f.name for f in fm.fontManager.ttflist}
    chosen = None
    for name in candidates:
        if name in all_names:
            chosen = name
            plt.rcParams['font.sans-serif'] = [name]
            break
    plt.rcParams['axes.unicode_minus'] = False
    plt.rcParams['figure.dpi'] = 130
    plt.rcParams['font.size'] = 11
    return chosen

FONT_NAME = setup_chinese_font()
print('使用中文字体：', FONT_NAME)


def style_standard():
    plt.rcParams.update({
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.grid': True,
        'grid.alpha': 0.22,
        'axes.facecolor': 'white',
        'figure.facecolor': 'white',
        'axes.titleweight': 'semibold',
    })


def style_enhanced():
    plt.rcParams.update({
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.grid': True,
        'grid.alpha': 0.18,
        'axes.facecolor': '#FAFBFC',
        'figure.facecolor': 'white',
        'axes.titleweight': 'bold',
    })


def savefig_both(fig, out_base: Path):
    fig.savefig(out_base.with_suffix('.png'), dpi=300, bbox_inches='tight')
    fig.savefig(out_base.with_suffix('.pdf'), bbox_inches='tight')
    plt.close(fig)
    print('已导出：', out_base.name)

使用中文字体： Microsoft YaHei


In [16]:

# ---------- 导入当前 q2 模块 ----------
def import_module_from_file(path: Path, module_name: str):
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

q2 = import_module_from_file(Q2_PY, 'q2_lambda_main')
q2_penalty = import_module_from_file(Q2_PENALTY_PY, 'q2_lambda_penalty_ref')

print('已导入：', q2)
print('当前 q2.py 中冻结预测参数 FORECAST =')
print(json.dumps(q2.FORECAST, ensure_ascii=False, indent=2))
print('q2_with_penalty.py 中的终端参数：')
print('  TERMINAL_TARGET_KWH =', getattr(q2_penalty, 'TERMINAL_TARGET_KWH', '未找到'))
print('  TERMINAL_PENALTY_YUAN_PER_KWH =', getattr(q2_penalty, 'TERMINAL_PENALTY_YUAN_PER_KWH', '未找到'))

已导入： <module 'q2_lambda_main' from 'e:\\桌面\\新建文件夹\\P2_SOFT_PENALTY_PACKAGE\\q2.py'>
当前 q2.py 中冻结预测参数 FORECAST =
{
  "window_days": 60,
  "k_load": 3,
  "k_pv": 2,
  "alpha_load": 0.3,
  "alpha_pv": 10.0,
  "scenario_days": 28
}
q2_with_penalty.py 中的终端参数：
  TERMINAL_TARGET_KWH = 6000.0
  TERMINAL_PENALTY_YUAN_PER_KWH = 0.45



## 四、数据一致性核验

本节只做两件事：

1. 检查原始附件与当前预处理 CSV 是否一致；
2. 检查电价附件与当前固定电价 CSV 是否一致。

这样可以保证后续敏感性分析确实是建立在**当前认可的数据版本**之上。


In [17]:

# ---------- 原始附件与预处理 CSV 一致性核验 ----------
raw_dates, raw_load, raw_pv, raw_prices = q2.load_inputs(ACTUAL_XLSX, PRICE_XLSX)
csv_dates, csv_load, csv_pv, csv_prices = q2.load_inputs(ACTUAL_CSV, PRICE_CSV)

assert len(raw_dates) == len(csv_dates) == 365, '日期长度不一致'
assert np.all(raw_dates == csv_dates), '原始附件与 CSV 日期不一致'

max_diff_load = float(np.max(np.abs(raw_load - csv_load)))
max_diff_pv = float(np.max(np.abs(raw_pv - csv_pv)))
max_diff_price = float(np.max(np.abs(raw_prices - csv_prices)))

check_df = pd.DataFrame([
    ['负荷数据', max_diff_load, '通过' if max_diff_load < 1e-12 else '失败'],
    ['光伏数据', max_diff_pv, '通过' if max_diff_pv < 1e-12 else '失败'],
    ['电价数据', max_diff_price, '通过' if max_diff_price < 1e-12 else '失败'],
], columns=['核验对象', '最大绝对差值', '结论'])

display(check_df)

assert max_diff_load < 1e-12, '负荷数据不一致'
assert max_diff_pv < 1e-12, '光伏数据不一致'
assert max_diff_price < 1e-12, '电价数据不一致'

DATES, LOAD, PV, PRICES = csv_dates, csv_load, csv_pv, csv_prices

print('一致性核验通过：后续全部基于当前 CSV 重算。')

,核验对象,最大绝对差值,结论
0,负荷数据,0.0,通过
1,光伏数据,0.0,通过
2,电价数据,0.0,通过


一致性核验通过：后续全部基于当前 CSV 重算。



## 五、重新生成滚动预测结果与场景库

虽然本 Notebook 的重点是 \(\lambda\) 敏感性分析，但为了保证逻辑闭环，仍然先按当前 `q2.py` 的因果滚动方式重新生成：

- 点预测结果；
- 残差场景库；
- 开发期与运行期的预测误差摘要。

这一步不读取任何旧的预测结果文件。


In [18]:

# ---------- 重新生成点预测与场景 ----------
pred_l, pred_p, SCENARIOS = q2.forecast_scenarios(DATES, LOAD, PV)

rows = []
for d in range(len(DATES)):
    if np.isnan(pred_l[d]).all() or np.isnan(pred_p[d]).all():
        continue
    for t in range(144):
        rows.append({
            'date': str(pd.Timestamp(DATES[d]).date()),
            'slot': t + 1,
            'actual_load_kw': float(LOAD[d, t]),
            'pred_load_kw': float(pred_l[d, t]),
            'actual_pv_kw': float(PV[d, t]),
            'pred_pv_kw': float(pred_p[d, t]),
            'actual_net_kw': float(LOAD[d, t] - PV[d, t]),
            'pred_net_kw': float(pred_l[d, t] - pred_p[d, t]),
        })
point_df = pd.DataFrame(rows)
point_df['date'] = pd.to_datetime(point_df['date'])
point_df.to_csv(OUT / '01_point_forecast.csv', index=False, encoding='utf-8-sig')

DEV_MASK = (point_df['date'] >= pd.Timestamp(q2.DEV_START)) & (point_df['date'] <= pd.Timestamp(q2.DEV_END))
RUN_MASK = (point_df['date'] >= pd.Timestamp(q2.RUN_START)) & (point_df['date'] <= pd.Timestamp(q2.RUN_END))


def metrics_block(df, prefix):
    out = {}
    for tag, a, p in [
        ('load', 'actual_load_kw', 'pred_load_kw'),
        ('pv', 'actual_pv_kw', 'pred_pv_kw'),
        ('net', 'actual_net_kw', 'pred_net_kw'),
    ]:
        err = df[p] - df[a]
        out[f'{prefix}_{tag}_mae'] = float(np.mean(np.abs(err)))
        out[f'{prefix}_{tag}_rmse'] = float(np.sqrt(np.mean(err**2)))
    return out

forecast_metrics = {}
forecast_metrics.update(metrics_block(point_df.loc[DEV_MASK], 'dev'))
forecast_metrics.update(metrics_block(point_df.loc[RUN_MASK], 'run'))
forecast_metrics_df = pd.DataFrame([forecast_metrics]).T.reset_index()
forecast_metrics_df.columns = ['metric', 'value']
forecast_metrics_df.to_csv(OUT / '02_forecast_metrics.csv', index=False, encoding='utf-8-sig')
display(forecast_metrics_df)

,metric,value
0,dev_load_mae,139.604395
1,dev_load_rmse,181.086848
2,dev_pv_mae,192.984601
3,dev_pv_rmse,353.975273
4,dev_net_mae,272.735595
5,dev_net_rmse,392.623240
6,run_load_mae,120.758252
7,run_load_rmse,159.274428
8,run_pv_mae,156.390993
9,run_pv_rmse,302.518439



## 六、\(\lambda\) 敏感性分析：完整重跑逻辑

为了保证逻辑充分严谨，本 Notebook 对每一个候选 \(\lambda\) 都执行以下过程：

1. 固定预测层（即滚动 PCA + Ridge 预测结果不变）；
2. 在开发期（5–8 月）对风险分位数 \(q\in\{0.05,0.10,\ldots,0.95\}\) **重新扫描**；
3. 取该 \(\lambda\) 下的最优风险分位数 \(q^*(\lambda)\)；
4. 在运行期（2–12 月）用 \(q^*(\lambda)\) 与该 \(\lambda\) 重新回放最终策略；
5. 统计：
   - 实际总费用；
   - 计划购电费用；
   - 紧急购电费用；
   - 紧急购电总量 / 天数；
   - 日末储能的均值、最小值、最大值；
   - 日度费用与日度日末储能分布。

这种做法比“固定 \(q=0.85\) 再扫 \(\lambda\)”更严格，也更适合写入论文。


In [19]:

# ---------- 对每个 lambda 重新扫描 q* 并回放最终策略 ----------
DEV_IDX = np.where((DATES >= pd.Timestamp(q2.DEV_START)) & (DATES <= pd.Timestamp(q2.DEV_END)))[0]
RUN_IDX = np.where((DATES >= pd.Timestamp(q2.RUN_START)) & (DATES <= pd.Timestamp(q2.RUN_END)))[0]

summary_rows = []
scan_rows = []
daily_all = []
slots_all = []

for lam in LAMBDA_GRID:
    q_star, scan = q2.scan_q_star(
        DATES, LOAD, PV, PRICES, SCENARIOS,
        terminal_target_kwh=E_TARGET,
        terminal_penalty_yuan_per_kwh=float(lam),
    )
    scan = scan.copy()
    scan['lambda'] = float(lam)
    scan_rows.append(scan)

    daily, slots = q2.run_q(
        DATES, LOAD, PV, PRICES, SCENARIOS,
        RUN_IDX, float(q_star), keep_slots=True,
        terminal_target_kwh=E_TARGET,
        terminal_penalty_yuan_per_kwh=float(lam),
    )
    daily = daily.copy()
    slots = slots.copy()
    daily['date'] = pd.to_datetime(daily['date'])
    slots['date'] = pd.to_datetime(slots['date'])
    daily['lambda'] = float(lam)
    slots['lambda'] = float(lam)

    end_soc = slots.groupby('date')['planned_storage_end_kwh'].last()
    summary_rows.append({
        'lambda': float(lam),
        'q_star': float(q_star),
        'planned_cost_yuan': float(daily['planned_cost_yuan'].sum()),
        'emergency_cost_yuan': float(daily['emergency_cost_yuan'].sum()),
        'total_cost_yuan': float(daily['total_cost_yuan'].sum()),
        'emergency_kwh': float(daily['emergency_kwh'].sum()),
        'unused_kwh': float(daily['unused_kwh'].sum()),
        'emergency_days': int((daily['emergency_kwh'] > 1e-7).sum()),
        'avg_terminal_kwh': float(end_soc.mean()),
        'min_terminal_kwh': float(end_soc.min()),
        'max_terminal_kwh': float(end_soc.max()),
        'daily_cost_std': float(daily['total_cost_yuan'].std(ddof=0)),
        'daily_terminal_std': float(end_soc.std(ddof=0)),
    })
    daily_all.append(daily)
    slots_all.append(slots)
    print(f"lambda={lam:.2f} 完成，q*={q_star:.2f}, total={daily['total_cost_yuan'].sum():.2f}")

summary_df = pd.DataFrame(summary_rows).sort_values('lambda').reset_index(drop=True)
scan_df = pd.concat(scan_rows, ignore_index=True)
daily_all_df = pd.concat(daily_all, ignore_index=True)
slots_all_df = pd.concat(slots_all, ignore_index=True)

summary_df['total_cost_wan'] = summary_df['total_cost_yuan'] / 1e4
summary_df['planned_cost_wan'] = summary_df['planned_cost_yuan'] / 1e4
summary_df['emergency_cost_wan'] = summary_df['emergency_cost_yuan'] / 1e4

summary_df.to_csv(OUT / '03_lambda_summary.csv', index=False, encoding='utf-8-sig')
scan_df.to_csv(OUT / '04_lambda_q_scan_long.csv', index=False, encoding='utf-8-sig')
daily_all_df.to_csv(OUT / '05_lambda_daily_long.csv', index=False, encoding='utf-8-sig')
slots_all_df.to_csv(OUT / '06_lambda_slots_long.csv', index=False, encoding='utf-8-sig')

display(summary_df.round(4))

lambda=0.00 完成，q*=0.85, total=14343442.93
lambda=0.10 完成，q*=0.85, total=14343442.93
lambda=0.20 完成，q*=0.85, total=14343442.93
lambda=0.30 完成，q*=0.85, total=14343442.93
lambda=0.45 完成，q*=0.85, total=14331509.57
lambda=0.60 完成，q*=0.85, total=14333072.09
lambda=0.80 完成，q*=0.85, total=14333072.09
lambda=1.00 完成，q*=0.85, total=14333072.09


,lambda,q_star,planned_cost_yuan,emergency_cost_yuan,total_cost_yuan,emergency_kwh,unused_kwh,emergency_days,avg_terminal_kwh,min_terminal_kwh,max_terminal_kwh,daily_cost_std,daily_terminal_std,total_cost_wan,planned_cost_wan,emergency_cost_wan
0,0.00,0.85,1.346696e+07,876478.6501,1.434344e+07,225320.9902,3.040670e+06,333,1200.0,1200.0,1200.0,14965.0363,0.0,1434.3443,1346.6964,87.6479
1,0.10,0.85,1.346696e+07,876478.6501,1.434344e+07,225320.9902,3.040670e+06,333,1200.0,1200.0,1200.0,14965.0363,0.0,1434.3443,1346.6964,87.6479
2,0.20,0.85,1.346696e+07,876478.6501,1.434344e+07,225320.9902,3.040670e+06,333,1200.0,1200.0,1200.0,14965.0363,0.0,1434.3443,1346.6964,87.6479
3,0.30,0.85,1.346696e+07,876478.6501,1.434344e+07,225320.9902,3.040670e+06,333,1200.0,1200.0,1200.0,14965.0363,0.0,1434.3443,1346.6964,87.6479
4,0.45,0.85,1.345503e+07,876478.6501,1.433151e+07,225320.9902,3.040670e+06,333,1950.0,1950.0,1950.0,14962.4009,0.0,1433.1510,1345.5031,87.6479
5,0.60,0.85,1.345645e+07,876617.3743,1.433307e+07,225378.9732,3.040728e+06,333,6000.0,6000.0,6000.0,14904.1833,0.0,1433.3072,1345.6455,87.6617
6,0.80,0.85,1.345645e+07,876617.3743,1.433307e+07,225378.9732,3.040728e+06,333,6000.0,6000.0,6000.0,14904.1833,0.0,1433.3072,1345.6455,87.6617
7,1.00,0.85,1.345645e+07,876617.3743,1.433307e+07,225378.9732,3.040728e+06,333,6000.0,6000.0,6000.0,14904.1833,0.0,1433.3072,1345.6455,87.6617


In [20]:

# ---------- 根据敏感性分析结果自动选择真正的最优 lambda ----------
# 0.45 仅作为当前模型中的“参考值”，不再被强制选中。
reference_lambda = 0.45

best_idx = summary_df["total_cost_yuan"].idxmin()
chosen_lambda = float(summary_df.loc[best_idx, "lambda"])
chosen_q = float(summary_df.loc[best_idx, "q_star"])

chosen_daily = daily_all_df[daily_all_df["lambda"] == chosen_lambda].copy()
chosen_slots = slots_all_df[slots_all_df["lambda"] == chosen_lambda].copy()
chosen_events = q2.emergency_events(chosen_slots)

result2_path = OUT / "result2_lambda_best.xlsx"
q2.export_result2(
    TEMPLATE_XLSX,
    result2_path,
    chosen_daily,
    chosen_slots,
    chosen_events,
)

reference_row = summary_df[np.isclose(summary_df["lambda"], reference_lambda)]
reference_total = (
    float(reference_row["total_cost_yuan"].iloc[0])
    if len(reference_row) else np.nan
)

summary_record = {
    "best_lambda": chosen_lambda,
    "best_q_star": chosen_q,
    "reference_lambda": reference_lambda,
    "reference_lambda_total_cost_yuan": reference_total,
    "terminal_target_kwh": float(E_TARGET),
    "planned_cost_yuan": float(chosen_daily["planned_cost_yuan"].sum()),
    "emergency_cost_yuan": float(chosen_daily["emergency_cost_yuan"].sum()),
    "total_cost_yuan": float(chosen_daily["total_cost_yuan"].sum()),
    "emergency_kwh": float(chosen_daily["emergency_kwh"].sum()),
    "unused_kwh": float(chosen_daily["unused_kwh"].sum()),
}

(OUT / "07_best_lambda_summary.json").write_text(
    json.dumps(summary_record, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("扫描集合内真正的最优 λ =", chosen_lambda)
print("对应 q* =", chosen_q)
print("最优实际总费用 =", summary_record["total_cost_yuan"])
if np.isfinite(reference_total):
    print(
        f"参考 λ=0.45 的实际总费用 = {reference_total:.2f} 元，"
        f"相对最优多花 {reference_total-summary_record['total_cost_yuan']:.2f} 元"
    )
print("已按真正最优 λ 导出：", result2_path)

扫描集合内真正的最优 λ = 0.45
对应 q* = 0.85
最优实际总费用 = 14331509.571149785
参考 λ=0.45 的实际总费用 = 14331509.57 元，相对最优多花 0.00 元
已按真正最优 λ 导出： e:\桌面\新建文件夹\P2_SOFT_PENALTY_PACKAGE\fresh_outputs_lambda\result2_lambda_best.xlsx



## 七、辅助表与代表性日期自动选择

为了让图更像优秀论文中的成体系图池，下面会自动构造几张辅助表：

- 每个 \(\lambda\) 下的日末储能分布表；
- 每个 \(\lambda\) 下的日度费用分布表；
- 代表性机制对比日（默认选择：在最终采用的 \(\lambda\) 下，紧急购电量最大的那一天）。


In [21]:

# ---------- 辅助数据表 ----------
terminal_daily_df = (
    slots_all_df
    .groupby(['lambda', 'date'], as_index=False)
    .agg(day_end_kwh=('planned_storage_end_kwh', 'last'))
)

cost_daily_df = daily_all_df[['lambda', 'date', 'planned_cost_yuan', 'emergency_cost_yuan', 'total_cost_yuan', 'emergency_kwh']].copy()

rep_lambda = chosen_lambda
rep_day = pd.to_datetime(
    daily_all_df[daily_all_df['lambda'] == rep_lambda]
    .sort_values(['emergency_kwh', 'total_cost_yuan'], ascending=[False, False])
    ['date'].iloc[0]
)
print('代表性对比日：', rep_day.date(), '（依据：在 λ=', rep_lambda, ' 下紧急购电量最大）')

代表性对比日： 2025-06-01 （依据：在 λ= 0.45  下紧急购电量最大）



## 八、A 类：规范论文版图集

A 类图强调：
- 清晰；
- 克制；
- 标题与中文表达规范；
- 适合直接插入论文正文。


In [22]:

# ---------- A 类图池 ----------
style_standard()

# A01 不同 lambda 下全年实际总费用变化（主结论折线图）
fig, ax = plt.subplots(figsize=(8.4, 4.9))
ax.plot(summary_df['lambda'], summary_df['total_cost_wan'], marker='o', linewidth=2.3)
ax.axvline(chosen_lambda, linestyle='--', linewidth=1.3, alpha=0.8)
row = summary_df[summary_df['lambda'] == chosen_lambda].iloc[0]
ax.scatter([chosen_lambda], [row['total_cost_wan']], s=90, zorder=5)
ax.annotate(f"扫描最优：λ={chosen_lambda:.2f}\n总费用={row['total_cost_wan']:.2f} 万元",
            xy=(chosen_lambda, row['total_cost_wan']), xytext=(chosen_lambda+0.05, row['total_cost_wan']+0.4),
            arrowprops=dict(arrowstyle='->', lw=1.1), bbox=dict(boxstyle='round,pad=0.32', fc='white', ec='0.7'))
ax.set_title('不同终端惩罚系数下全年实际总费用变化')
ax.set_xlabel('终端惩罚系数 λ（元/kWh）')
ax.set_ylabel('全年实际总费用（万元）')
ax.set_xticks(summary_df['lambda'])
plt.tight_layout(); savefig_both(fig, FIG_A / 'A01_lambda_vs_total_cost')

# A02 费用分解多折线图
fig, ax = plt.subplots(figsize=(8.7, 5.0))
ax.plot(summary_df['lambda'], summary_df['planned_cost_wan'], marker='o', linewidth=2.0, label='计划购电费用')
ax.plot(summary_df['lambda'], summary_df['emergency_cost_wan'], marker='s', linewidth=2.0, label='紧急购电费用')
ax.plot(summary_df['lambda'], summary_df['total_cost_wan'], marker='^', linewidth=2.3, label='实际总费用')
ax.axvline(chosen_lambda, linestyle='--', linewidth=1.3, alpha=0.8)
ax.set_title('不同 λ 下费用构成变化')
ax.set_xlabel('终端惩罚系数 λ（元/kWh）')
ax.set_ylabel('费用（万元）')
ax.legend(frameon=False)
ax.set_xticks(summary_df['lambda'])
plt.tight_layout(); savefig_both(fig, FIG_A / 'A02_lambda_cost_decomposition')

# A03 柱线组合：总费用 + 平均日末储能
fig, ax1 = plt.subplots(figsize=(8.8, 5.0))
ax1.bar(summary_df['lambda'].astype(str), summary_df['avg_terminal_kwh'], alpha=0.75)
ax1.set_xlabel('终端惩罚系数 λ（元/kWh）')
ax1.set_ylabel('平均日末储能（kWh）')
ax2 = ax1.twinx()
ax2.plot(summary_df['lambda'].astype(str), summary_df['total_cost_wan'], marker='o', linewidth=2.1)
ax2.set_ylabel('全年实际总费用（万元）')
ax1.set_title('不同 λ 下平均日末储能与全年总费用')
plt.tight_layout(); savefig_both(fig, FIG_A / 'A03_lambda_terminal_and_total')

# A04 柱线组合：紧急购电总量 + 紧急购电天数
fig, ax1 = plt.subplots(figsize=(8.8, 5.0))
ax1.bar(summary_df['lambda'].astype(str), summary_df['emergency_kwh'], alpha=0.75)
ax1.set_xlabel('终端惩罚系数 λ（元/kWh）')
ax1.set_ylabel('全年紧急购电量（kWh）')
ax2 = ax1.twinx()
ax2.plot(summary_df['lambda'].astype(str), summary_df['emergency_days'], marker='o', linewidth=2.1)
ax2.set_ylabel('发生紧急购电的天数（天）')
ax1.set_title('不同 λ 下紧急购电量与紧急购电天数')
plt.tight_layout(); savefig_both(fig, FIG_A / 'A04_lambda_emergency_volume_days')

# A05 箱线图：每日日末储能分布
fig, ax = plt.subplots(figsize=(9.0, 5.0))
labels = [f'{lam:.2f}' for lam in LAMBDA_GRID]
data = [terminal_daily_df.loc[terminal_daily_df['lambda'] == lam, 'day_end_kwh'].to_numpy() for lam in LAMBDA_GRID]
ax.boxplot(data, labels=labels, showfliers=False)
ax.set_title('不同 λ 下每日日末储能分布')
ax.set_xlabel('终端惩罚系数 λ（元/kWh）')
ax.set_ylabel('日末储能（kWh）')
plt.tight_layout(); savefig_both(fig, FIG_A / 'A05_lambda_terminal_boxplot')

# A06 箱线图：每日实际费用分布
fig, ax = plt.subplots(figsize=(9.0, 5.0))
data = [cost_daily_df.loc[cost_daily_df['lambda'] == lam, 'total_cost_yuan'].to_numpy()/1e4 for lam in LAMBDA_GRID]
ax.boxplot(data, labels=labels, showfliers=False)
ax.set_title('不同 λ 下每日实际费用分布')
ax.set_xlabel('终端惩罚系数 λ（元/kWh）')
ax.set_ylabel('每日实际费用（万元）')
plt.tight_layout(); savefig_both(fig, FIG_A / 'A06_lambda_daily_cost_boxplot')

# A07 热力图：lambda-q 开发期总费用
pivot = scan_df.pivot(index='lambda', columns='q', values='total_cost_yuan') / 1e4
fig, ax = plt.subplots(figsize=(10.0, 5.5))
im = ax.imshow(pivot.to_numpy(), aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{x:.2f}' for x in pivot.columns], rotation=45)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f'{x:.2f}' for x in pivot.index])
ax.set_xlabel('风险分位数 q')
ax.set_ylabel('终端惩罚系数 λ（元/kWh）')
ax.set_title('开发期“λ-q”组合下实际总费用热力图（万元）')
fig.colorbar(im, ax=ax, shrink=0.88, label='开发期实际总费用（万元）')
plt.tight_layout(); savefig_both(fig, FIG_A / 'A07_lambda_q_heatmap')

# A08 折线图：lambda 与 q* / 平均日末储能（双轴）
fig, ax1 = plt.subplots(figsize=(8.8, 4.9))
ax1.plot(summary_df['lambda'], summary_df['q_star'], marker='o', linewidth=2.0)
ax1.set_xlabel('终端惩罚系数 λ（元/kWh）')
ax1.set_ylabel('对应最优风险分位数 q*')
ax2 = ax1.twinx()
ax2.plot(summary_df['lambda'], summary_df['avg_terminal_kwh'], marker='s', linewidth=2.0)
ax2.set_ylabel('平均日末储能（kWh）')
ax1.set_title('不同 λ 下最优风险分位数与平均日末储能')
plt.tight_layout(); savefig_both(fig, FIG_A / 'A08_lambda_qstar_terminal')

# A09 机制曲线图：代表性日期的 SOC 轨迹
fig, ax = plt.subplots(figsize=(10.2, 4.8))
for lam in COMPARE_LAMBDAS:
    sub = slots_all_df[(slots_all_df['lambda'] == lam) & (slots_all_df['date'] == rep_day)].sort_values('slot')
    ax.plot(sub['slot'], sub['planned_storage_end_kwh'], linewidth=2.0, label=f'λ={lam:.2f}')
ax.axhline(E_TARGET, linestyle='--', linewidth=1.0, alpha=0.8)
ax.set_title(f'代表性日期（{rep_day.date()}）在不同 λ 下的储能轨迹')
ax.set_xlabel('10 分钟时段')
ax.set_ylabel('储能量（kWh）')
ax.legend(frameon=False, ncol=3)
plt.tight_layout(); savefig_both(fig, FIG_A / 'A09_repday_storage_trajectory')

# A10 机制曲线图：代表性日期的计划购电与紧急购电
fig, ax = plt.subplots(figsize=(10.2, 4.8))
for lam in COMPARE_LAMBDAS:
    sub = slots_all_df[(slots_all_df['lambda'] == lam) & (slots_all_df['date'] == rep_day)].sort_values('slot')
    ax.plot(sub['slot'], sub['planned_grid_kwh']*6, linewidth=1.8, label=f'计划购电功率等效 λ={lam:.2f}')
ax.set_title(f'代表性日期（{rep_day.date()}）在不同 λ 下的计划购电功率')
ax.set_xlabel('10 分钟时段')
ax.set_ylabel('功率（kW）')
ax.legend(frameon=False, ncol=2)
plt.tight_layout(); savefig_both(fig, FIG_A / 'A10_repday_plan_grid')

已导出： A01_lambda_vs_total_cost
已导出： A02_lambda_cost_decomposition
已导出： A03_lambda_terminal_and_total
已导出： A04_lambda_emergency_volume_days
已导出： A05_lambda_terminal_boxplot
已导出： A06_lambda_daily_cost_boxplot
已导出： A07_lambda_q_heatmap
已导出： A08_lambda_qstar_terminal
已导出： A09_repday_storage_trajectory
已导出： A10_repday_plan_grid



## 九、B 类：增强展示版图集

B 类图与 A 类图传达的信息保持一致，但会进一步强调：
- 最优 / 最终采用点；
- 低成本区域；
- 关键对比；
- 代表性机制差异。


In [23]:

# ---------- B 类图池 ----------
style_enhanced()

# B01 主结论增强版：总费用
fig, ax = plt.subplots(figsize=(8.8, 5.1))
y = summary_df['total_cost_wan'].to_numpy()
ax.plot(summary_df['lambda'], y, marker='o', linewidth=2.6)
ax.fill_between(summary_df['lambda'], y, y.max()+0.8, alpha=0.08)
chosen_row_plot = summary_df[summary_df['lambda'] == chosen_lambda].iloc[0]
ax.scatter([chosen_lambda], [chosen_row_plot['total_cost_wan']], s=120, zorder=6)
ax.axvline(chosen_lambda, linestyle='--', linewidth=1.4, alpha=0.85)
ax.annotate(f"扫描最优 λ={chosen_lambda:.2f}\n总费用={chosen_row_plot['total_cost_wan']:.2f} 万元",
            xy=(chosen_lambda, chosen_row_plot['total_cost_wan']), xytext=(chosen_lambda+0.08, y.min()+0.35),
            arrowprops=dict(arrowstyle='->', lw=1.2),
            bbox=dict(boxstyle='round,pad=0.35', fc='white', ec='0.65'))
ax.set_title('终端惩罚系数 λ 对全年实际总费用的影响（增强版）')
ax.set_xlabel('终端惩罚系数 λ（元/kWh）')
ax.set_ylabel('全年实际总费用（万元）')
ax.set_xticks(summary_df['lambda'])
plt.tight_layout(); savefig_both(fig, FIG_B / 'B01_lambda_vs_total_cost')

# B02 费用分解增强版
fig, ax = plt.subplots(figsize=(9.0, 5.2))
ax.plot(summary_df['lambda'], summary_df['planned_cost_wan'], marker='o', linewidth=2.1, label='计划购电费用')
ax.plot(summary_df['lambda'], summary_df['emergency_cost_wan'], marker='s', linewidth=2.1, label='紧急购电费用')
ax.plot(summary_df['lambda'], summary_df['total_cost_wan'], marker='^', linewidth=2.6, label='实际总费用')
ax.fill_between(summary_df['lambda'], summary_df['planned_cost_wan'], summary_df['total_cost_wan'], alpha=0.06)
ax.axvline(chosen_lambda, linestyle='--', linewidth=1.4, alpha=0.85)
ax.set_title('不同 λ 下费用构成变化（增强版）')
ax.set_xlabel('终端惩罚系数 λ（元/kWh）')
ax.set_ylabel('费用（万元）')
ax.legend(frameon=False)
plt.tight_layout(); savefig_both(fig, FIG_B / 'B02_lambda_cost_decomposition')

# B03 平均日末储能 + 总费用增强版
fig, ax1 = plt.subplots(figsize=(8.9, 5.1))
ax1.bar(summary_df['lambda'].astype(str), summary_df['avg_terminal_kwh'], alpha=0.80)
ax1.set_xlabel('终端惩罚系数 λ（元/kWh）')
ax1.set_ylabel('平均日末储能（kWh）')
ax2 = ax1.twinx()
ax2.plot(summary_df['lambda'].astype(str), summary_df['total_cost_wan'], marker='o', linewidth=2.2)
ax2.set_ylabel('全年实际总费用（万元）')
ax1.set_title('平均日末储能与全年总费用的联合变化（增强版）')
plt.tight_layout(); savefig_both(fig, FIG_B / 'B03_lambda_terminal_and_total')

# B04 紧急购电量 + 紧急购电天数增强版
fig, ax1 = plt.subplots(figsize=(8.9, 5.1))
ax1.bar(summary_df['lambda'].astype(str), summary_df['emergency_kwh'], alpha=0.80)
ax1.set_xlabel('终端惩罚系数 λ（元/kWh）')
ax1.set_ylabel('全年紧急购电量（kWh）')
ax2 = ax1.twinx()
ax2.plot(summary_df['lambda'].astype(str), summary_df['emergency_days'], marker='o', linewidth=2.2)
ax2.set_ylabel('紧急购电天数（天）')
ax1.set_title('紧急购电强度与频次对 λ 的响应（增强版）')
plt.tight_layout(); savefig_both(fig, FIG_B / 'B04_lambda_emergency_volume_days')

# B05 终端储能箱线图增强版
fig, ax = plt.subplots(figsize=(9.3, 5.1))
data = [terminal_daily_df.loc[terminal_daily_df['lambda'] == lam, 'day_end_kwh'].to_numpy() for lam in LAMBDA_GRID]
box = ax.boxplot(data, labels=labels, showfliers=False, patch_artist=True)
for patch in box['boxes']:
    patch.set_alpha(0.45)
ax.set_title('不同 λ 下每日日末储能分布（增强版）')
ax.set_xlabel('终端惩罚系数 λ（元/kWh）')
ax.set_ylabel('日末储能（kWh）')
plt.tight_layout(); savefig_both(fig, FIG_B / 'B05_lambda_terminal_boxplot')

# B06 日费用箱线图增强版
fig, ax = plt.subplots(figsize=(9.3, 5.1))
data = [cost_daily_df.loc[cost_daily_df['lambda'] == lam, 'total_cost_yuan'].to_numpy()/1e4 for lam in LAMBDA_GRID]
box = ax.boxplot(data, labels=labels, showfliers=False, patch_artist=True)
for patch in box['boxes']:
    patch.set_alpha(0.45)
ax.set_title('不同 λ 下每日实际费用分布（增强版）')
ax.set_xlabel('终端惩罚系数 λ（元/kWh）')
ax.set_ylabel('每日实际费用（万元）')
plt.tight_layout(); savefig_both(fig, FIG_B / 'B06_lambda_daily_cost_boxplot')

# B07 热力图增强版
fig, ax = plt.subplots(figsize=(10.2, 5.6))
mat = pivot.to_numpy()
im = ax.imshow(mat, aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{x:.2f}' for x in pivot.columns], rotation=45)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f'{x:.2f}' for x in pivot.index])
ax.set_xlabel('风险分位数 q')
ax.set_ylabel('终端惩罚系数 λ（元/kWh）')
ax.set_title('开发期“λ-q”组合下实际总费用热力图（增强版）')
min_pos = np.unravel_index(np.argmin(mat), mat.shape)
ax.scatter([min_pos[1]], [min_pos[0]], marker='*', s=180, zorder=6)
fig.colorbar(im, ax=ax, shrink=0.88, label='开发期实际总费用（万元）')
plt.tight_layout(); savefig_both(fig, FIG_B / 'B07_lambda_q_heatmap')

# B08 代表性日储能轨迹增强版
fig, ax = plt.subplots(figsize=(10.5, 5.0))
for lam in COMPARE_LAMBDAS:
    sub = slots_all_df[(slots_all_df['lambda'] == lam) & (slots_all_df['date'] == rep_day)].sort_values('slot')
    ax.plot(sub['slot'], sub['planned_storage_end_kwh'], linewidth=2.2, label=f'λ={lam:.2f}')
ax.axhline(E_TARGET, linestyle='--', linewidth=1.0, alpha=0.8, label='参考终端储能 6000 kWh')
ax.text(0.01, 0.98, f'代表性日期：{rep_day.date()}\n按 λ={rep_lambda:.2f} 下紧急购电量最大自动选取', transform=ax.transAxes,
        ha='left', va='top', bbox=dict(boxstyle='round,pad=0.35', fc='white', ec='0.7', alpha=0.95))
ax.set_title('代表性日期下不同 λ 的储能轨迹对比（增强版）')
ax.set_xlabel('10 分钟时段')
ax.set_ylabel('储能量（kWh）')
ax.legend(frameon=False, ncol=2)
plt.tight_layout(); savefig_both(fig, FIG_B / 'B08_repday_storage_trajectory')

# B09 代表性日计划购电功率增强版
fig, ax = plt.subplots(figsize=(10.5, 5.0))
for lam in COMPARE_LAMBDAS:
    sub = slots_all_df[(slots_all_df['lambda'] == lam) & (slots_all_df['date'] == rep_day)].sort_values('slot')
    ax.plot(sub['slot'], sub['planned_grid_kwh']*6, linewidth=2.0, label=f'λ={lam:.2f}')
ax.set_title('代表性日期下不同 λ 的计划购电功率对比（增强版）')
ax.set_xlabel('10 分钟时段')
ax.set_ylabel('功率（kW）')
ax.legend(frameon=False, ncol=3)
plt.tight_layout(); savefig_both(fig, FIG_B / 'B09_repday_plan_grid')

# B10 代表性日紧急购电功率增强版
fig, ax = plt.subplots(figsize=(10.5, 5.0))
for lam in COMPARE_LAMBDAS:
    sub = slots_all_df[(slots_all_df['lambda'] == lam) & (slots_all_df['date'] == rep_day)].sort_values('slot')
    ax.plot(sub['slot'], sub['emergency_kwh']*6, linewidth=2.0, label=f'λ={lam:.2f}')
ax.set_title('代表性日期下不同 λ 的紧急购电功率对比（增强版）')
ax.set_xlabel('10 分钟时段')
ax.set_ylabel('功率（kW）')
ax.legend(frameon=False, ncol=3)
plt.tight_layout(); savefig_both(fig, FIG_B / 'B10_repday_emergency_grid')

已导出： B01_lambda_vs_total_cost
已导出： B02_lambda_cost_decomposition
已导出： B03_lambda_terminal_and_total
已导出： B04_lambda_emergency_volume_days
已导出： B05_lambda_terminal_boxplot
已导出： B06_lambda_daily_cost_boxplot
已导出： B07_lambda_q_heatmap
已导出： B08_repday_storage_trajectory
已导出： B09_repday_plan_grid
已导出： B10_repday_emergency_grid



## 十、结果摘要与论文写作提示

本节会自动输出一份简短摘要，便于你直接复制到报告或论文手稿中。你可以在此基础上再润色。


In [24]:

# ---------- 结果摘要与文本导出 ----------
best_idx = summary_df['total_cost_yuan'].idxmin()
best_row = summary_df.loc[best_idx]
chosen_row = summary_df[summary_df['lambda'] == chosen_lambda].iloc[0]

report_text = f"""# λ 敏感性分析摘要

## 1. 分析口径
- 预测模型固定为：W={q2.FORECAST['window_days']}, K_L={q2.FORECAST['k_load']}, K_P={q2.FORECAST['k_pv']}, α_L={q2.FORECAST['alpha_load']}, α_P={q2.FORECAST['alpha_pv']}。
- 终端参考储能取 E*={E_TARGET:.0f} kWh。
- 对每个 λ，均在开发期重新扫描 q∈{{0.05,0.10,...,0.95}}，取该 λ 下最优 q*(λ)，再在 2–12 月运行期回放最终策略。

## 2. 主要结论
- 在当前扫描集合中，总费用最低点对应 λ={best_row['lambda']:.2f}，其最优风险分位数为 q*={best_row['q_star']:.2f}。
- 该点的全年实际总费用为 {best_row['total_cost_yuan']:.2f} 元，其中计划购电费用为 {best_row['planned_cost_yuan']:.2f} 元，紧急购电费用为 {best_row['emergency_cost_yuan']:.2f} 元。
- 该点的平均日末储能为 {best_row['avg_terminal_kwh']:.2f} kWh。

## 3. 扫描最优参数
- 本文扫描最优 λ={chosen_row['lambda']:.2f}。
- 对应最优风险分位数 q*={chosen_row['q_star']:.2f}。
- 对应全年实际总费用为 {chosen_row['total_cost_yuan']:.2f} 元。
- 对应平均日末储能为 {chosen_row['avg_terminal_kwh']:.2f} kWh，紧急购电总量为 {chosen_row['emergency_kwh']:.2f} kWh。

## 4. 解释性结论
- 当 λ 较小时，模型更倾向于在日末释放储能，短视效应更明显；
- 随着 λ 增大，日末储能水平逐步上升，模型对未来时段的准备更充分；
- 但 λ 过大也会削弱调度灵活性，因此存在“成本—终端储能”之间的折中区间；
- 结合全年实际总费用与日末储能水平，λ={chosen_row['lambda']:.2f} 能较好兼顾经济性与运行稳定性。
"""

(OUT / '08_lambda_report.md').write_text(report_text, encoding='utf-8')
print(report_text)

# λ 敏感性分析摘要

## 1. 分析口径
- 预测模型固定为：W=60, K_L=3, K_P=2, α_L=0.3, α_P=10.0。
- 终端参考储能取 E*=6000 kWh。
- 对每个 λ，均在开发期重新扫描 q∈{0.05,0.10,...,0.95}，取该 λ 下最优 q*(λ)，再在 2–12 月运行期回放最终策略。

## 2. 主要结论
- 在当前扫描集合中，总费用最低点对应 λ=0.45，其最优风险分位数为 q*=0.85。
- 该点的全年实际总费用为 14331509.57 元，其中计划购电费用为 13455030.92 元，紧急购电费用为 876478.65 元。
- 该点的平均日末储能为 1950.00 kWh。

## 3. 扫描最优参数
- 本文扫描最优 λ=0.45。
- 对应最优风险分位数 q*=0.85。
- 对应全年实际总费用为 14331509.57 元。
- 对应平均日末储能为 1950.00 kWh，紧急购电总量为 225320.99 kWh。

## 4. 解释性结论
- 当 λ 较小时，模型更倾向于在日末释放储能，短视效应更明显；
- 随着 λ 增大，日末储能水平逐步上升，模型对未来时段的准备更充分；
- 但 λ 过大也会削弱调度灵活性，因此存在“成本—终端储能”之间的折中区间；
- 结合全年实际总费用与日末储能水平，λ=0.45 能较好兼顾经济性与运行稳定性。



In [25]:

# ---------- 输出文件清单 ----------
print('\\nA 类图文件：')
for p in sorted(FIG_A.glob('*')):
    print(' -', p.name)

print('\\nB 类图文件：')
for p in sorted(FIG_B.glob('*')):
    print(' -', p.name)

print('\\n数值结果文件：')
for p in sorted(OUT.glob('*')):
    print(' -', p.name)

\nA 类图文件：
 - A01_lambda_vs_total_cost.pdf
 - A01_lambda_vs_total_cost.png
 - A02_lambda_cost_decomposition.pdf
 - A02_lambda_cost_decomposition.png
 - A03_lambda_terminal_and_total.pdf
 - A03_lambda_terminal_and_total.png
 - A04_lambda_emergency_volume_days.pdf
 - A04_lambda_emergency_volume_days.png
 - A05_lambda_terminal_boxplot.pdf
 - A05_lambda_terminal_boxplot.png
 - A06_lambda_daily_cost_boxplot.pdf
 - A06_lambda_daily_cost_boxplot.png
 - A07_lambda_q_heatmap.pdf
 - A07_lambda_q_heatmap.png
 - A08_lambda_qstar_terminal.pdf
 - A08_lambda_qstar_terminal.png
 - A09_repday_storage_trajectory.pdf
 - A09_repday_storage_trajectory.png
 - A10_repday_plan_grid.pdf
 - A10_repday_plan_grid.png
\nB 类图文件：
 - B01_lambda_vs_total_cost.pdf
 - B01_lambda_vs_total_cost.png
 - B02_lambda_cost_decomposition.pdf
 - B02_lambda_cost_decomposition.png
 - B03_lambda_terminal_and_total.pdf
 - B03_lambda_terminal_and_total.png
 - B04_lambda_emergency_volume_days.pdf
 - B04_lambda_emergency_volume_days.png
